In [1]:
!pip install transformers==4.40.0 datasets evaluate rouge-score bert-score sentencepiece accelerate tqdm pandas numpy scikit-learn torch torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.9 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=fda3dcd0ec3c72d6c1e2377b399b80ba39cbe527a5702903897fa79bc8ab0bcb
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.1

In [2]:
import importlib, sys
required = ["transformers","evaluate","rouge_score","bert_score","sentencepiece","sklearn","tqdm","pandas","numpy","torch"]
missing  = [p for p in required if importlib.util.find_spec(p.replace("-","_")) is None]
if missing:
    print(f"Missing: {missing}  — run:  pip install {' '.join(missing)}")
else:
    print("✅ All packages present")


✅ All packages present


In [13]:
import os, re, json, random, warnings
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    get_linear_schedule_with_warmup,
)
import evaluate
from bert_score import score as bert_score_fn

warnings.filterwarnings("ignore")

# ── MPS (Apple Silicon) → CUDA → CPU ──
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("🍎 Apple Silicon MPS")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("⚡ CUDA GPU")
else:
    DEVICE = torch.device("cpu")
    print("💻 CPU")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")

⚡ CUDA GPU
PyTorch 2.10.0+cu128  |  device: cuda


In [14]:
# ============================================================
# CELL 2 — Load & Validate Dataset
# Place  NewsSumm_PP_flat.csv  in the same folder as this notebook
# ============================================================

CSV_PATH = "/content/NewsSumm_PP_flat.csv"

df = pd.read_csv(CSV_PATH)
# Use original-case article_text for better BART tokenisation
# Fall back to clean_article if article_text is missing
df["article"] = df["article_text"].fillna(df["clean_article"]).astype(str).str.strip()
df["summary"] = df["clean_summary"].astype(str).str.strip()

# Quality filter
df = df[
    (df["article"].str.split().str.len() >= 20) &   # article has substance
    (df["summary"].str.split().str.len() >= 5)  &   # summary is real
    (df["summary"].str.split().str.len() <= 80)     # cap runaway summaries
].reset_index(drop=True)

print(f"Clean pairs     : {len(df):,}")
print(f"Avg article len : {df['article'].str.split().str.len().mean():.0f} words")
print(f"Avg summary len : {df['summary'].str.split().str.len().mean():.0f} words")
print(f"Categories      : {sorted(df['new_category'].unique())}")

# ── Train / Val / Test split (stratified by category) ──
from sklearn.model_selection import train_test_split

train_df, tmp_df = train_test_split(df, test_size=0.15, random_state=SEED,
                                    stratify=df["new_category"])
val_df,  test_df = train_test_split(tmp_df, test_size=0.5, random_state=SEED,
                                    stratify=tmp_df["new_category"])

print(f"\nSplit → train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")

Clean pairs     : 43,781
Avg article len : 92 words
Avg summary len : 13 words
Categories      : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Split → train: 37,213  val: 3,284  test: 3,284


In [15]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — Tokeniser & PyTorch Dataset
# ─────────────────────────────────────────────────────────────

MODEL_NAME   = "facebook/bart-large-cnn"
MAX_INPUT    = 256   # articles are ~95 words → 256 tokens is safe and fast
MAX_TARGET   = 64    # summaries are ~15 words

print(f"Loading tokeniser from {MODEL_NAME} ...")
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
print("✅ Tokeniser ready")

class NewsDataset(Dataset):
    def __init__(self, dataframe):
        self.articles  = dataframe["article"].tolist()
        self.summaries = dataframe["summary"].tolist()

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.articles[idx],
            max_length=MAX_INPUT,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        with tokenizer.as_target_tokenizer():
            dec = tokenizer(
                self.summaries[idx],
                max_length=MAX_TARGET,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )
        labels = dec.input_ids.squeeze()
        labels[labels == tokenizer.pad_token_id] = -100   # ignore padding in loss
        return {
            "input_ids"      : enc.input_ids.squeeze(),
            "attention_mask" : enc.attention_mask.squeeze(),
            "labels"         : labels,
        }

# DataLoaders
# pin_memory only for CUDA — causes errors on MPS
pin = (DEVICE.type == "cuda")

BATCH_SIZE  = 4    # fits BART-large in 16 GB M2 unified memory
train_loader = DataLoader(NewsDataset(train_df), batch_size=BATCH_SIZE,
                          shuffle=True,  pin_memory=pin, num_workers=0)
val_loader   = DataLoader(NewsDataset(val_df),   batch_size=BATCH_SIZE,
                          shuffle=False, pin_memory=pin, num_workers=0)

print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")


Loading tokeniser from facebook/bart-large-cnn ...
✅ Tokeniser ready
Train batches : 9,304
Val batches   : 821


In [16]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — Model Setup
#
# Fine-tuning strategy for M2 Mac:
#   ✅ BART encoder — FROZEN  (already knows news text encoding)
#   ✅ BART decoder — TRAINABLE (learns Indian news style)
#   ✅ lm_head     — TRAINABLE (output vocabulary projection)
#
# This reduces trainable params from ~400 M → ~170 M
# and cuts memory + training time roughly in half.
# ─────────────────────────────────────────────────────────────

print(f"Loading {MODEL_NAME} ...")
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

# Freeze encoder
for p in model.model.encoder.parameters():
    p.requires_grad = False

# Make sure decoder + lm_head are trainable
for p in model.model.decoder.parameters():
    p.requires_grad = True
for p in model.lm_head.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Params: {trainable/1e6:.0f}M trainable / {total/1e6:.0f}M total ({100*trainable/total:.0f}%)")

model = model.to(DEVICE)
print(f"Model on {DEVICE} ✅")

Loading facebook/bart-large-cnn ...
Params: 254M trainable / 406M total (63%)
Model on cuda ✅


In [17]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — Fine-Tuning
# Expected time on M2 Mac: ~15–25 min for 3 epochs
# ─────────────────────────────────────────────────────────────

EPOCHS      = 3
GRAD_ACCUM  = 4       # effective batch = 4 × 4 = 16
LR          = 3e-5
WARMUP      = 0.06    # 6% warmup steps

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=0.01)

total_steps  = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps = int(total_steps * WARMUP)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"Epochs          : {EPOCHS}")
print(f"Effective batch : {BATCH_SIZE * GRAD_ACCUM}")
print(f"Optimiser steps : {total_steps:,}  (warmup: {warmup_steps:,})")
print()

history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    # ── TRAIN ──
    model.train()
    epoch_loss, steps = 0.0, 0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f"Epoch {epoch}/{EPOCHS} [train]")

    for i, batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        try:
            out  = model(**batch)
            loss = out.loss / GRAD_ACCUM
            loss.backward()
            epoch_loss += out.loss.item()
            steps      += 1
        except RuntimeError as e:
            print(f"  batch {i} skipped: {e}")
            optimizer.zero_grad()
            continue

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        if steps % 50 == 0:
            pbar.set_postfix({"loss": f"{epoch_loss / max(steps,1):.4f}"})

    # ── VALIDATE ──
    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [val]  ", leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            try:
                out       = model(**batch)
                val_loss += out.loss.item()
                val_steps += 1
            except: continue

    avg_train = epoch_loss / max(steps, 1)
    avg_val   = val_loss   / max(val_steps, 1)
    history["train_loss"].append(avg_train)
    history["val_loss"].append(avg_val)

    print(f"Epoch {epoch} → train: {avg_train:.4f}  val: {avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "cfms_best.pt")
        print(f"  ✅ Best model saved  (val_loss={best_val_loss:.4f})")

print("\n✅ Training complete")

Epochs          : 3
Effective batch : 16
Optimiser steps : 6,978  (warmup: 418)



Epoch 1/3 [train]: 100%|██████████| 9304/9304 [1:08:03<00:00,  2.28it/s, loss=2.1724]


Epoch 1 → train: 2.1723  val: 2.0351
  ✅ Best model saved  (val_loss=2.0351)


Epoch 2/3 [train]: 100%|██████████| 9304/9304 [1:07:58<00:00,  2.28it/s, loss=1.6117]


Epoch 2 → train: 1.6116  val: 2.0048
  ✅ Best model saved  (val_loss=2.0048)


Epoch 3/3 [train]: 100%|██████████| 9304/9304 [1:07:58<00:00,  2.28it/s, loss=1.3309]
                                                                    

Epoch 3 → train: 1.3310  val: 2.0250

✅ Training complete


In [18]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — Load Best Checkpoint + Generation Function
# ─────────────────────────────────────────────────────────────

model.load_state_dict(torch.load("cfms_best.pt", map_location=DEVICE))
model.eval()
print("✅ Best checkpoint loaded")

def summarise(text: str,
              num_beams: int  = 4,
              max_new_tokens: int = 60,
              min_length: int     = 8,
              length_penalty: float = 1.0) -> str:
    """Generate a summary for a single article."""
    inputs = tokenizer(
        text,
        max_length=MAX_INPUT,
        truncation=True,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_length=min_length,
            num_beams=num_beams,
            no_repeat_ngram_size=3,
            length_penalty=length_penalty,
            early_stopping=True,
        )
    return tokenizer.decode(ids[0], skip_special_tokens=True)

# Quick sanity check
sample_art = test_df.iloc[0]["article"]
sample_ref = test_df.iloc[0]["summary"]
print(f"\nArticle : {sample_art[:200]}...")
print(f"Reference: {sample_ref}")
print(f"Generated: {summarise(sample_art)}")

✅ Best checkpoint loaded

Article : NEW DELHI: PM Narendra Modi on Sunday said the entire world celebrated the 10th International Day of Yoga with great enthusiasm on June 21 while noting that locals had joined him in the yoga programme...
Reference: in radio address, pm highlights yoga enthusiasm across the globe
Generated: world celebrates yoga day with great enthusiasm, says pm modi


In [19]:
# ─────────────────────────────────────────────────────────────
# CELL 7 — Evaluate on Test Set
# ─────────────────────────────────────────────────────────────

rouge_metric = evaluate.load("rouge")

def compute_scores(preds, refs):
    r = rouge_metric.compute(predictions=preds, references=refs, use_stemmer=True)
    try:
        _, _, F1 = bert_score_fn(
            preds, refs,
            lang="en",
            model_type="distilbert-base-uncased",
            batch_size=32,
            device=str(DEVICE),
            verbose=False,
        )
        bs = round(F1.mean().item(), 4)
    except Exception as e:
        print(f"  BERTScore skipped: {e}")
        bs = 0.0
    return {
        "ROUGE-1"   : round(r["rouge1"], 4),
        "ROUGE-2"   : round(r["rouge2"], 4),
        "ROUGE-L"   : round(r["rougeL"], 4),
        "BERTScore" : bs,
    }

# Generate for full test set
TEST_LIMIT  = len(test_df)   # evaluate every test example
predictions = []
references  = []

print(f"Generating summaries for {TEST_LIMIT} test articles...")
for _, row in tqdm(test_df.head(TEST_LIMIT).iterrows(), total=TEST_LIMIT):
    pred = summarise(row["article"])
    predictions.append(pred)
    references.append(row["summary"])

scores = compute_scores(predictions, references)

print("\n" + "="*50)
print("  TEST SET SCORES")
print("="*50)
for k, v in scores.items():
    bar = "█" * int(v * 40)
    print(f"  {k:12s} {v:.4f}  {bar}")
print("="*50)

Generating summaries for 3284 test articles...


100%|██████████| 3284/3284 [17:43<00:00,  3.09it/s]



  TEST SET SCORES
  ROUGE-1      0.4567  ██████████████████
  ROUGE-2      0.2551  ██████████
  ROUGE-L      0.4001  ████████████████
  BERTScore    0.8397  █████████████████████████████████


In [21]:
# ─────────────────────────────────────────────────────────────
# CELL 8 — 📰 GENERATED SUMMARIES
# ─────────────────────────────────────────────────────────────

CAT_NAMES = {0:"Entertainment/Sports", 1:"News Digest", 2:"Politics", 3:"Economy", 4:"Finance", 5:"Incidents"}

# Show 25 examples spread across categories
SHOW = 25
rows = test_df.head(TEST_LIMIT).reset_index(drop=True)

print("\n" + "▓"*68)
print("  CFMS — GENERATED NEWS SUMMARIES")
print("▓"*68)

for i, (pred, ref) in enumerate(zip(predictions[:SHOW], references[:SHOW])):
    row = rows.iloc[i]
    cat = CAT_NAMES.get(int(row.get("new_category", 0)), "General")

    print(f"\n{'─'*68}")
    print(f"  #{i+1:02d}  [{cat}]")
    print(f"{'─'*68}")
    print(f"  📰 ARTICLE:")
    print(f"     {row['article'][:220].strip()}{'...' if len(row['article'])>220 else ''}")
    print(f"\n  ✅ GENERATED SUMMARY:")
    print(f"     {pred}")
    print(f"\n  📋 REFERENCE:")
    print(f"     {ref}")

print("\n" + "▓"*68)
print("  FINAL SCORES")
print("▓"*68)
for k, v in scores.items():
    print(f"  {k:12s}: {v:.4f}")
print("▓"*68)
print(f"\n  Evaluated on {TEST_LIMIT:,} test articles")
print("▓"*68)


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  CFMS — GENERATED NEWS SUMMARIES
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

────────────────────────────────────────────────────────────────────
  #01  [News Digest]
────────────────────────────────────────────────────────────────────
  📰 ARTICLE:
     NEW DELHI: PM Narendra Modi on Sunday said the entire world celebrated the 10th International Day of Yoga with great enthusiasm on June 21 while noting that locals had joined him in the yoga programme in Srinagar.

"As t...

  ✅ GENERATED SUMMARY:
     world celebrates yoga day with great enthusiasm, says pm modi

  📋 REFERENCE:
     in radio address, pm highlights yoga enthusiasm across the globe

────────────────────────────────────────────────────────────────────
  #02  [News Digest]
────────────────────────────────────────────────────────────────────
  📰 ARTICLE:
     MUMBAI: A special court here has convicted a former chief manager of t

In [22]:
# ─────────────────────────────────────────────────────────────
# CELL 9 — Summarise Any Article
# Paste any news article below and get an instant summary
# ─────────────────────────────────────────────────────────────

MY_ARTICLE = """
India's cricket team secured a historic victory in the ICC Champions Trophy 2025
final against New Zealand at the Dubai International Cricket Stadium on Sunday.
Captain Rohit Sharma led the team to a commanding 8-wicket win, chasing down
the target of 252 with 14 balls to spare. Virat Kohli remained unbeaten on 82
as India clinched the title for the third time. The win was met with celebrations
across India, with fans taking to streets in Mumbai, Delhi and Kolkata.
Prime Minister Modi congratulated the team, calling it a proud moment for the nation.
"""

print("Article:")
print(MY_ARTICLE.strip())
print()
print("Generated Summary:")
print(summarise(MY_ARTICLE.strip(), min_length=10, max_new_tokens=60))

Article:
India's cricket team secured a historic victory in the ICC Champions Trophy 2025
final against New Zealand at the Dubai International Cricket Stadium on Sunday.
Captain Rohit Sharma led the team to a commanding 8-wicket win, chasing down
the target of 252 with 14 balls to spare. Virat Kohli remained unbeaten on 82
as India clinched the title for the third time. The win was met with celebrations
across India, with fans taking to streets in Mumbai, Delhi and Kolkata.
Prime Minister Modi congratulated the team, calling it a proud moment for the nation.

Generated Summary:
india clinches historic victory in icc champions trophy 2025 final against new zealand
